In [1]:
import numpy as np
import torch

In [2]:
edge_name_list = ['party_party', 'party_prox5', 'party_prox2', 'party_focal', 'prox5_prox5', \
    'prox2_prox5', 'prox5_focal', 'prox2_prox2','prox2_focal', 'grooming']

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_steps = 8
val_steps = 3
test_steps = 3
max_epochs = 5000
early_stop_epochs = 3000
lr = 0.1
sim_coeff = 1.0
deg_coeff = 1.0
reg_coeff = 0.0 # 0.001
start_year = 1998
name_str = list(np.load('../data/chimps_names.npy'))
num_chimps = len(name_str)

In [4]:
# Define a function to format an array to three decimal places
def format_array(array):
    return [f"{num:.1f}" for num in array]

In [5]:
seed_list = [0, 10, 20]
def get_mean_std_deviations(initial_params_all):
    initial_values = initial_params_all[:-1]
    initial_addition = initial_params_all[-1]
    parameters_full = np.ones((len(seed_list), len(initial_params_all)+1))
    test_cost_full = np.ones(len(seed_list))
    has_res = False
    for ind, seed in enumerate(seed_list):
        for val_ind in range(len(initial_values)):
            if initial_values[val_ind] == 0:
                initial_values[val_ind] = 0.0001 # do not have actual zero
        if initial_addition == 0:
            initial_addition = 0.0001 # do not have actual zero
        elif initial_addition == 1:
            initial_addition = 0.9999 # do not have actual one
        initial_values_str = ''.join(str(e) for e in initial_values)
        args_info = (f'trainSteps{train_steps}_val{val_steps}_test{test_steps}'
                f'_maxEpochs{max_epochs}_early{early_stop_epochs}_lr{lr}_seed{seed}'
                    f'initVal{initial_values_str}_initAdd{initial_addition}'
                    f'_simCoeff{sim_coeff}_degCoeff{deg_coeff}_regCoeff{reg_coeff}')
        
        try:
            parameters_full[ind, 1:] = np.load('../saved_parameters/chimps/'+args_info+'_parameters.npy')
            test_cost_full[ind] = np.load('../saved_epoch_cost/chimps/'+args_info+'_epoch_cost.npy')[-1]
            has_res = True
        except FileNotFoundError:
            parameters_full[ind] = np.nan
            test_cost_full[ind] = np.nan
    if has_res:
        parameters_mean = np.nanmean(parameters_full, axis=0)
        parameters_std = np.nanstd(parameters_full, axis=0)
        test_cost_mean = np.nanmean(test_cost_full)
        test_cost_std = np.nanstd(test_cost_full)

        initial_parameter_values = np.ones_like(parameters_mean)
        initial_parameter_values[1:-1] = initial_values
        initial_parameter_values[-1] = initial_addition

        deviations_from_initial = parameters_mean - initial_parameter_values
        return parameters_mean, parameters_std, deviations_from_initial, test_cost_mean, test_cost_std
    else:
        return None, None, None, None, None

#### Analysis on test costs and parameters

In [6]:
initial_params_list = [[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
                       [0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2],
                       [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
                       [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
                       [2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 1.0],
                       [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 1.0],
                       [1.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
                       [0.1, 1.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
                       [0.1, 0.1, 1.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
                       [0.1, 0.1, 0.1, 1.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
                       [0.1, 0.1, 0.1, 0.1, 1.0, 0.1, 0.1, 0.1, 0.1, 0.1],
                       [0.1, 0.1, 0.1, 0.1, 0.1, 1.0, 0.1, 0.1, 0.1, 0.1],
                       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 1.0, 0.1, 0.1, 0.1],
                       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 1.0, 0.1, 0.1],
                       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 1.0, 0.1],
                       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 1.0]
]

In [7]:
for initial_param in initial_params_list:
    parameters_mean, parameters_std, deviations_from_initial, test_cost_mean, test_cost_std = get_mean_std_deviations(initial_param)
    if parameters_mean is not None:
        print(f'Initial values: {format_array(initial_param)}')
        print(f'Mean parameters: {format_array(parameters_mean)}')
        print(f'Std parameters: {format_array(parameters_std)}')
        print(f'Deviations from initial: {format_array(deviations_from_initial)}')
        print(f'Mean test cost: {test_cost_mean:.6f}')
        print(f'Std test cost: {test_cost_std:.6f}')
        print('\n')

Initial values: ['0.1', '0.1', '0.1', '0.1', '0.1', '0.1', '0.1', '0.1', '0.1', '0.1']
Mean parameters: ['1.0', '0.0', '0.0', '0.0', '33.1', '0.0', '0.1', '34.9', '0.0', '0.0', '0.0']
Std parameters: ['0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0']
Deviations from initial: ['0.0', '-0.1', '-0.1', '-0.1', '33.0', '-0.1', '-0.0', '34.8', '-0.1', '-0.1', '-0.1']
Mean test cost: 0.007555
Std test cost: 0.000000


Initial values: ['0.2', '0.2', '0.2', '0.2', '0.2', '0.2', '0.2', '0.2', '0.2', '0.2']
Mean parameters: ['1.0', '0.0', '0.0', '0.0', '33.3', '0.0', '0.1', '34.8', '0.0', '0.0', '0.0']
Std parameters: ['0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0']
Deviations from initial: ['0.0', '-0.2', '-0.2', '-0.2', '33.1', '-0.2', '-0.1', '34.6', '-0.2', '-0.2', '-0.2']
Mean test cost: 0.007557
Std test cost: 0.000000


Initial values: ['0.5', '0.5', '0.5', '0.5', '0.5', '0.5', '0.5', '0.5', '0.5', '0.5']
Mean parameters: ['1.0', '0.0

#### Final results to print for Chimps

In [15]:
selected_increments = np.array([1, 0.0, 0.0, 0.0, 4.7, 1.3, 1.6, 2.0, 0.0, 0.1, 0.0])
for i, edge_key in enumerate(edge_name_list+['consec_add']):
    if edge_key != 'grooming':
        printed_edge_key = edge_key[:5] + '\\' + edge_key[5:]
    else:
        printed_edge_key = edge_key
    if i == 10:
        print('consecutive addition: ${:.1f}$.'.format(selected_increments[i]))
    else:
        print('{}: ${:.1f}$,'.format(printed_edge_key, selected_increments[:(i+1)].sum()), end=' ')

party\_party: $1.0$, party\_prox5: $1.0$, party\_prox2: $1.0$, party\_focal: $1.0$, prox5\_prox5: $5.7$, prox2\_prox5: $7.0$, prox5\_focal: $8.6$, prox2\_prox2: $10.6$, prox2\_focal: $10.6$, grooming: $10.7$, consecutive addition: $0.0$.
